# MUSIC-Based 2D Radar Imaging

Demonstrates high-resolution angle-of-arrival imaging using the MUSIC algorithm on a Vayyar-style Uniform Planar Array (20 TX × 20 RX).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[0]))

import numpy as np
import torch
import matplotlib.pyplot as plt
from witwin.radar import Radar
from witwin.radar.sigproc import MUSICImager


## 1. Radar Config — Vayyar-style 20×20 UPA

In [ ]:
fc = 77e9

config = {
    "num_tx": 20, "num_rx": 20,
    "fc": fc,
    "slope": 60.012,
    "adc_samples": 256,
    "adc_start_time": 6,
    "sample_rate": 4400,
    "idle_time": 7,
    "ramp_end_time": 65,
    "chirp_per_frame": 8,
    "frame_per_second": 10,
    "num_doppler_bins": 8,
    "num_range_bins": 256,
    "num_angle_bins": 64,
    "power": 15,
    "tx_loc": [[i, 0, 0] for i in range(20)],
    "rx_loc": [[20, -i, 0] for i in range(20)],
}
radar = Radar(config)

## 2. Scene: Two Points at Different Angles

In [ ]:
points = np.array([
    [-0.5, 0, -3],   # left target
    [0.5,  0, -3],   # right target
], dtype=np.float32)

velocity = np.array([0, 0, 0.01])

def location_function(t):
    pos = torch.tensor(points + velocity * t, dtype=torch.float32, device='cuda')
    intensity = torch.ones(pos.shape[0], dtype=torch.float32, device='cuda')
    return intensity, pos

## 3. Generate Radar Frame (20×20 MIMO)

In [ ]:
frame = radar.mimo(location_function, t0=0)
print(f"Frame shape: {frame.shape}  (TX, RX, chirps, ADC)")

## 4. MUSIC Imaging

In [ ]:
imager = MUSICImager(num_tx=20, num_rx=20, num_signals=7,
                     spatial_smooth=3, num_pixels=128, num_chirps=8)

image3D = imager.radar_image(frame)
print(f"Image shape: {image3D.shape}  (H, W, range_bins)")

# Sum across range bins for a 2D angle image
image2D = image3D.sum(dim=2)

## 5. Visualization

In [ ]:
ranges = radar.ranges.cpu().numpy()
velocities = radar.velocities.cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (a) Range-Doppler (physical axes)
fft2d = torch.fft.fftshift(torch.fft.fft2(frame[0, 0, :, :].cpu()), dim=0)
rd_mag = 20 * torch.log10(torch.abs(fft2d) + 1e-6).numpy()
axes[0].imshow(
    rd_mag[:, :len(ranges)],
    extent=[ranges[0], ranges[-1], velocities[0], velocities[-1]],
    aspect='auto', origin='lower', cmap='jet',
)
axes[0].set_xlabel('Range (m)')
axes[0].set_ylabel('Velocity (m/s)')
axes[0].set_title('Range-Doppler (TX0, RX0)')

# (b) MUSIC 2D image
music_db = 20 * torch.log10(torch.abs(image2D) + 1e-6)
im = axes[1].imshow(np.rot90(music_db.detach().cpu().numpy(), 1), cmap='jet')
axes[1].set_title('MUSIC 2D Angle Image')
axes[1].set_xlabel('Azimuth')
axes[1].set_ylabel('Elevation')
plt.colorbar(im, ax=axes[1])

# (c) MUSIC per range bin (3D slices)
for i in range(image3D.shape[2]):
    slice_db = 20 * torch.log10(torch.abs(image3D[:, :, i]) + 1e-6)
    axes[2].plot(slice_db.sum(dim=0).detach().cpu().numpy(), label=f'bin {i}')
axes[2].set_title('MUSIC azimuth profile per range bin')
axes[2].set_xlabel('Azimuth pixel')
axes[2].set_ylabel('Power (dB)')
axes[2].legend(fontsize=6)

plt.tight_layout()